# Procesamiento y Análisis de Datos EMG

Este cuaderno realiza el procesamiento de las sesiones almacenadas en `../storage/sessions`.
El flujo de trabajo incluye:
1. **Carga de Datos**: Lectura de archivos CSV.
2. **Análisis Exploratorio**: Conteo de registros por etiqueta.
3. **Preprocesamiento**: Limpieza básica.
4. **Normalización**: Escalado de las señales EMG para facilitar su comparación y análisis.

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt

# Configuración de gráficos
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Ruta al directorio de sesiones
sessions_dir = os.path.join('..', 'storage', 'sessions')

csv_files = glob.glob(os.path.join(sessions_dir, '**', '*.csv'), recursive=True)
print(f"Se encontraron {len(csv_files)} archivos de sesión.")

## 1. Carga y Combinación de Datos

In [ ]:
dfs = []

for file in csv_files:
    try:
        df = pd.read_csv(file)
        # Identificador de sesión para agrupar/normalizar por separado si es necesario
        df['source_file'] = os.path.basename(file)
        dfs.append(df)
    except Exception as e:
        print(f"Error leyendo {file}: {e}")

if dfs:
    full_df = pd.concat(dfs, ignore_index=True)
    print(f"Total de registros cargados: {len(full_df)}")
    display(full_df.head())
else:
    full_df = pd.DataFrame()

In [ ]:
# Definir canales EMG
emg_channels = ['emg1', 'emg2', 'emg3']

# Filtrar solo columnas existentes
available_channels = [c for c in emg_channels if c in full_df.columns]
print(f"Canales disponibles: {available_channels}")

## 2. Análisis por Agrupación (GroupBy)
Contamos cuántos registros hay por cada etiqueta (gesto).

In [ ]:
if not full_df.empty and 'labels' in full_df.columns:
    grouped = full_df.groupby('labels')[available_channels].count()
    display(grouped)
    
    # Gráfico de conteo
    grouped.plot(kind='bar', figsize=(10, 5), title="Registros por Gesto")
    plt.ylabel("Cantidad de Muestras")
    plt.show()

## 3. Normalización de Señales

Para comparar señales de diferentes sesiones o sujetos, es crucial normalizarlas.
Aquí aplicaremos una **Normalización Min-Max**, que escala los valores al rango `[0, 1]`.

$$ x_{norm} = \frac{x - \min(x)}{\max(x) - \min(x)} $$

> **Nota:** La normalización se aplica **por sesión** (`source_file`), ya que la amplitud de la señal EMG depende de la colocación de los electrodos y la conductividad de la piel, que varía entre sesiones.

In [ ]:
def normalize_minmax(group, channels):
    # Copia del grupo para no afectar el original inmediatamente
    group_norm = group.copy()
    
    for channel in channels:
        col_min = group[channel].min()
        col_max = group[channel].max()
        
        # Evitar división por cero si la señal es plana
        if col_max - col_min == 0:
            group_norm[f'{channel}_norm'] = 0
        else:
            group_norm[f'{channel}_norm'] = (group[channel] - col_min) / (col_max - col_min)
            
    return group_norm

if not full_df.empty:
    # Aplicar normalización agrupando por archivo de origen (sesión)
    full_df_norm = full_df.groupby('source_file', group_keys=False).apply(lambda x: normalize_minmax(x, available_channels))
    
    print("Normalización completada. Nuevas columnas generadas:")
    print([c for c in full_df_norm.columns if '_norm' in c])
    display(full_df_norm.head())

## 4. Visualización: Original vs Normalizada
Comparamos la señal 'cruda' con la señal normalizada de una sesión de ejemplo.

In [ ]:
if not full_df_norm.empty:
    # Seleccionar una sesión para visualizar (la primera disponible)
    sample_session = full_df_norm['source_file'].unique()[0]
    subset = full_df_norm[full_df_norm['source_file'] == sample_session].reset_index(drop=True)
    
    # Canal a visualizar
    channel_to_plot = available_channels[0] # Por ejemplo emg1
    
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    
    # Señal Original
    axes[0].plot(subset[channel_to_plot], color='#1f77b4', alpha=0.8)
    axes[0].set_title(f"Señal Original ({channel_to_plot}) - Sesión: {sample_session}")
    axes[0].set_ylabel("Amplitud (uV o ADC)")
    
    # Señal Normalizada
    axes[1].plot(subset[f'{channel_to_plot}_norm'], color='#2ca02c', alpha=0.8)
    axes[1].set_title(f"Señal Normalizada ({channel_to_plot}) [0-1]")
    axes[1].set_ylabel("Amplitud Normalizada")
    axes[1].set_xlabel("Muestras")
    
    plt.tight_layout()
    plt.show()